# NOAA sea level: separate seasonal variation and trend

What changes when 300 monthly relative water levels become yearly means or a seasonal heatmap? Use the same station and datum throughout; a local trend is not an inundation forecast.

## Objectives

- Check month completeness, station, datum and units
- Construct line, seasonal and annual comparison views
- Treat a fitted slope as a descriptive model of this selection

## Start in a fresh kernel

Run every code cell from the top. The Book shows publication-time outputs;
JupyterLite uses the Python (Pyodide) kernel and downloads its runtime and
packages initially. All observations and metadata below come from packaged
snapshots: no live API, token, paid service or GPU is required. If a file is
missing or modified, validation fails explicitly; restore the supplied files.
Download notebook edits and exports to keep them outside browser storage.
The Python code is visible so that transformations can be inspected.

In [ ]:
import sys
from pathlib import Path

if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib"])

# Book starts here; desktop/Lite may start at either the notebook or content root.
candidates = [Path.cwd(), Path.cwd() / "repository_examples_v2",
              Path.cwd() / "repository_examples",
              Path.cwd() / "content" / "repository_examples"]
lesson_dir = next((p for p in candidates if (p / "repo_examples.py").is_file()), None)
if lesson_dir is None:
    raise FileNotFoundError("Open this lesson from content/repository_examples on desktop, or repository_examples_v2 in Lite, with repo_examples.py and data/ alongside it.")
sys.path.insert(0, str(lesson_dir))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
import importlib
import repo_examples as rx
# Pick up a published helper correction when rerunning in an existing kernel.
rx = importlib.reload(rx)

if sys.platform != "emscripten":
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (9, 4.8), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
display(rx.validate_all())

## Evidence and attribution

The cards describe the evidence type before any visualization. The manifest retains exact URLs/queries, retrieval UTC, SHA-256 checksums, field order, row counts and transformation recipes. Dataset terms remain separate from the undecided project license.

In [ ]:
display(rx.source_card("noaa"))

In [ ]:
water = rx.read_csv("noaa/monthly.csv")
raw = rx.read_json("noaa/raw.json")
display(raw["metadata"])
water["date"] = pd.to_datetime(water[["year", "month"]].assign(day=1))
assert len(water) == 300 and water.date.is_unique
assert water.groupby("year").size().eq(12).all()
assert np.allclose(water.msl_m, [float(r["MSL"]) for r in raw["data"]])
display(water.head())
display(water.inferred.value_counts(dropna=False).rename("monthly_records"))

## View 1: preserve monthly variability

Monthly means already aggregate high-frequency tides and weather effects; the extrema fields have different meanings and should not be drawn as confidence intervals. The trailing 12-month mean requires twelve records and lags turning points. No missing months have been interpolated.

**Method connection:** the catalog's Line chart (§5) and Table (§3) provide a starting point; small multiples and seasonal heatmaps add comparisons across temporal scales. A water-level curve does not encode population exposure, facility risk or storm probability. The related Sea-Level-Rise repository supplies narrative and comparison patterns; its fictional exposure inputs are excluded.

In [ ]:
fig, ax = plt.subplots()
ax.plot(water.date, water.msl_m, color="#8aa5b0", lw=1, label="Monthly mean")
ax.plot(water.date, water.msl_m.rolling(12, min_periods=12).mean(), color="#075e7c", lw=2, label="Trailing 12-month mean")
rx.finish(ax, "Sewells Point 8638610 — relative water level", "Month (GMT)", "Water level (m, station MSL datum)")
ax.legend()
fig.tight_layout()
plt.show()

## View 2: season and year in separate directions

The same months align vertically in this matrix, making seasonal recurrence easier to see than in the line. Color represents water level in meters, rather than a rate of sea-level rise. A fixed scale prevents each year from looking equally variable.

In [ ]:
grid = water.pivot(index="year", columns="month", values="msl_m")
fig, axes = plt.subplots(1, 2, figsize=(11, 6), gridspec_kw={"width_ratios": [2, 1]})
im = axes[0].imshow(grid, aspect="auto", cmap="viridis", vmin=water.msl_m.min(), vmax=water.msl_m.max())
axes[0].set(xticks=range(12), xticklabels=range(1, 13), yticks=range(0, 25, 4), yticklabels=grid.index[::4],
            xlabel="Calendar month", ylabel="Year", title="Monthly mean water level (m, MSL)")
season = water.groupby("month").msl_m.agg(["mean", "min", "max"])
axes[1].plot(season["mean"], season.index, "o-", color="#075e7c")
axes[1].hlines(season.index, season["min"], season["max"], color="#8aa5b0", alpha=.8)
rx.finish(axes[1], "Observed min–max, not uncertainty", "Water level (m, MSL)", "Month")
fig.colorbar(im, ax=axes[0], label="m, MSL", shrink=.7)
plt.show()

## View 3: fit and inspect, rather than extrapolate

All years here have twelve months, so annual means give each year equal month coverage. An ordinary least-squares line summarizes this 25-year subset. Its slope is sensitive to the period and does not account for serial correlation, vertical land motion or other physical mechanisms. The residual panel shows what the line leaves behind; these residuals are not a calibrated future uncertainty interval.

In [ ]:
annual = water.groupby("year", as_index=False).msl_m.mean()
centered_year = annual.year - annual.year.mean()
slope, intercept = np.polyfit(centered_year, annual.msl_m, 1)
annual["fit_m"] = intercept + slope * centered_year
annual["residual_m"] = annual.msl_m - annual.fit_m
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
axes[0].plot(annual.year, annual.msl_m, "o", label="12-month annual mean")
axes[0].plot(annual.year, annual.fit_m, label=f"Subset OLS slope {1000*slope:.2f} mm/year")
axes[0].legend()
rx.finish(axes[0], "Descriptive fit, 2000–2024", "", "Water level (m, MSL)")
axes[1].axhline(0, color="black", lw=.7)
axes[1].bar(annual.year, annual.residual_m, color="#4f8c9d")
rx.finish(axes[1], "Residuals retain interannual variation", "Year", "Observed − fitted (m)")
fig.tight_layout()
plt.show()
display(annual.tail())

## Exercise

Refit the annual slope for 2005–2024 and 2010–2024. Report the units and period beside each slope, and explain why these are not the official NOAA long-term trend or a 2050 projection. Try a seasonal anomaly by subtracting each calendar month's sample mean; show which variation it removes.

## Limitations and reflection

These are local relative water levels at 36.9428° N, 76.3286° W, station 8638610, station MSL datum and GMT. Highest/lowest are retained source fields, not confidence bounds. Land movement, weather, tides and ocean dynamics contribute to local relative levels. There is no ground-elevation, exposure, population denominator or global reconstruction in this snapshot.

## Citations and repository connections

[NOAA fixed monthly query](https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?product=monthly_mean&application=JupyterLiteDataVisualization&begin_date=20000101&end_date=20241231&datum=MSL&station=8638610&time_zone=gmt&units=metric&format=json), [CO-OPS API definitions](https://api.tidesandcurrents.noaa.gov/api/dev), [NOS reuse guidance](https://oceanservice.noaa.gov/about/faq.html). Narrative reference: [Sea-Level-Rise at 6076bea2c878](https://github.com/jltobias/JupyterLite-Sea-Level-Rise/tree/6076bea2c8781285b204a5af0c3cd0ccc5d5ef8e).

See the [companion index](index.md) and [full attribution](https://jltobias.github.io/JupyterLite-Data-Visualization/book/references.html). These are original teaching code and narrative; linked repository notebooks and authored figures have not been copied.